In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import  RunnablePassthrough
from langchain_openai import ChatOpenAI

BASE_DIR = Path.cwd()
sudheer_report_file = BASE_DIR / "1_sudheer_real_estates_dubai.pdf"
print(sudheer_report_file)

pyPDFLoader = PyPDFLoader(sudheer_report_file).load()
print("pdf Content 📕: ",pyPDFLoader)
# For All Pages Metadata is same only change at page numbers
print("Metadata For Page 1 : ", pyPDFLoader[0].metadata)
#print("Metadata For Page 2 : ", pyPDFLoader[1].metadata)
#print("Metadata For Page 3 : ", pyPDFLoader[2].metadata)
#print("Metadata For Page 4 : ", pyPDFLoader[3].metadata)
#print("Metadata For Page 5 : ", pyPDFLoader[4].metadata)
print("Total pages : ", len(pyPDFLoader))
print("1st Page Content first 50 Characters : ", pyPDFLoader[0].page_content[:50],"........")
# To print Pages of All the content in pdf 
pdf_full_content = ''
#for i,page in enumerate(pyPDFLoader):
#    print("==========Page Number : ", i+1," ====================")
#    print(" page Content First 30 Characters : ",page.page_content[:30])
#    pdf_full_content += page.page_content
#print("Full Content 50 characters : ",pdf_full_content[:50])

chunk=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunk_data = chunk.split_documents(pyPDFLoader)
print(chunk_data)
len(chunk_data)
# Dummy Testing Chunk
#chunk_1 = RecursiveCharacterTextSplitter(chunk_size=3,chunk_overlap=2)
#chunk_data_1 = chunk_1.split_text("sudheer is a home. Home is at Hyderabad")
#print(chunk_data_1)
#print(len(chunk_data_1))
#print("Chunk Metadata : ",chunk_data[8].metadata)
load_dotenv()
index = faiss.IndexFlatL2(1536)
api_key = os.getenv("OPENAI_API_KEY")
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)
vector_store=FAISS(embedding_function=embedding,index=index,docstore=InMemoryDocstore(), index_to_docstore_id={} )
vector_store.add_documents(chunk_data)
retriever=vector_store.as_retriever(search_kwargs={"k":3})
#print(len(retriever.invoke("Quarterly Performance?")))
#retriever.invoke("Quarterly Performance?")[1].page_content

template = """
You are a helpful assistant.

Answer the question only using the information provided in the context.

Context:
{context}

Question:
{question}

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not in the context, say "I don't know."

Answer:
"""

promte=PromptTemplate.from_template(template)
print(promte)

def format_doc(docs):
    return "\n\n".join([doc.page_content for doc in docs])

myllm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

rag_chain= {"context": retriever|format_doc, "question": RunnablePassthrough() }|promte| myllm | StrOutputParser()
print(rag_chain)
#rag_chain.invoke("what is company name")
#rag_chain.invoke("which quarter has more sales?")
#rag_chain.invoke("total units sold in palm jumaria")
#rag_chain.invoke("Which area should I focus on year 2026?")
rag_chain.invoke("Total Transactions by Sudheer Real estates?")





d:\VS\sudheer_python\sudheer_python\RAG_Pipeline\1_sudheer_real_estates_dubai.pdf
pdf Content 📕:  [Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sudheer Real Estates - Annual Sales Report 2025', 'source': 'd:\\VS\\sudheer_python\\sudheer_python\\RAG_Pipeline\\1_sudheer_real_estates_dubai.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content="Executive Summary 2025: Sudheer Real Estates recorded its highest annual transaction volume in Dubai during\nFY2025, reaching total gross sales of AED 485.2 Million across 168 successful closed transactions. Strategic positioning\nin high-growth corridors—notably Downtown Jebel Ali and Downtown Dubai—drove over 50% of the firm's total revenue. \n1. KEY FINANCIAL HIGHLIGHTS\nTOTAL GROSS\nSALES\nAED\n485.2M\n↑ +24.8% YoY\nTRANSACTIONS\n168 Units\n↑ +18.2% Growth\nAVG SALE VALUE\nAED 2.88M\nLuxury Segment\nCOMMISSION\nREVENUE\nAED\n12.13M\n2.5% Avg Margin\n2. EXECUTIVE MARKET ANALYSIS

'Total transactions by Sudheer Real Estates are 168 units.'